# Notebook 10: Despliegue del Modelo (Inferencia)

**Objetivo:** Definir el flujo de inferencia del MLP entrenado para nuevas solicitudes de crédito. El flujo replica exactamente el preprocesamiento del entrenamiento (mismo escalamiento MinMax ajustado solo en Train) y produce la probabilidad de mora. Se valida con casos de prueba: caso válido, caso límite, entrada inválida y error de inferencia (Sección 17 de la plantilla).

In [1]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Clonar el repositorio para acceder a la carpeta src (si no existe)
if not os.path.exists('redes-neuronales-riesgo-crediticio'):
    !git clone https://github.com/JuanCardona2203/redes-neuronales-riesgo-crediticio.git

for repo_path in ['/content/redes-neuronales-riesgo-crediticio', '..']:
    if repo_path not in sys.path:
        sys.path.append(repo_path)

from src.model import CreditRiskMLP
from src.predict import predecir, VARIABLES

In [2]:
# Reconstrucción del escalador con el mismo criterio del entrenamiento (fit solo en Train)
SEED = 42
url_dataset = 'https://raw.githubusercontent.com/pr1m8/Credit-Default/main/credit_risk_dataset.csv'
df = pd.read_csv(url_dataset)[VARIABLES + ['loan_status']].copy()
df = df[df['person_age'] <= 100].copy()
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

X = df[VARIABLES]
y = df['loan_status']
X_train, _, _, _ = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)

scaler = MinMaxScaler().fit(X_train)
print('Escalador reconstruido (ajustado solo en Train).')

Escalador reconstruido (ajustado solo en Train).


In [3]:
# Carga del modelo entrenado
model = CreditRiskMLP(input_dim=6)
model.load_state_dict(torch.load('models/mlp_credit_risk.pt'))
model.eval()

# Flujo de inferencia: validación -> escalamiento -> red -> probabilidad
def evaluar_solicitud(solicitud):
    faltantes = [v for v in VARIABLES if v not in solicitud]
    if faltantes:
        raise ValueError(f'Faltan variables en la solicitud: {faltantes}')

    fila = pd.DataFrame([[solicitud[v] for v in VARIABLES]], columns=VARIABLES)
    if fila.isnull().any().any():
        raise ValueError('La solicitud contiene valores nulos.')

    X_esc = scaler.transform(fila)
    prob, pred = predecir(model, X_esc)
    return {
        'probabilidad_mora': round(float(prob[0]), 4),
        'decision': 'DENEGAR' if pred[0] == 1 else 'APROBAR'
    }

print('Flujo de inferencia listo.')

Flujo de inferencia listo.


In [4]:
# Caso válido: solicitante de bajo riesgo
caso_valido = {
    'person_age': 35, 'person_income': 80000, 'loan_amnt': 5000,
    'loan_int_rate': 7.5, 'loan_percent_income': 0.06, 'cb_person_cred_hist_length': 10
}
print('Caso válido (bajo riesgo):', evaluar_solicitud(caso_valido))

Caso válido (bajo riesgo): {'probabilidad_mora': 0.0221, 'decision': 'APROBAR'}


In [5]:
# Caso límite: solicitante de alto riesgo (alto endeudamiento y tasa elevada)
caso_limite = {
    'person_age': 22, 'person_income': 12000, 'loan_amnt': 8000,
    'loan_int_rate': 20.0, 'loan_percent_income': 0.67, 'cb_person_cred_hist_length': 2
}
print('Caso límite (alto riesgo):', evaluar_solicitud(caso_limite))

Caso límite (alto riesgo): {'probabilidad_mora': 0.9906, 'decision': 'DENEGAR'}


In [6]:
# Entrada inválida: falta una variable
caso_invalido = {
    'person_age': 30, 'person_income': 50000, 'loan_amnt': 10000,
    'loan_int_rate': 11.0, 'loan_percent_income': 0.20
}
try:
    evaluar_solicitud(caso_invalido)
except ValueError as e:
    print('Entrada inválida controlada:', e)

Entrada inválida controlada: Faltan variables en la solicitud: ['cb_person_cred_hist_length']


In [7]:
# Error de inferencia: valor nulo en una variable
caso_error = {
    'person_age': 30, 'person_income': 50000, 'loan_amnt': 10000,
    'loan_int_rate': np.nan, 'loan_percent_income': 0.20, 'cb_person_cred_hist_length': 5
}
try:
    evaluar_solicitud(caso_error)
except ValueError as e:
    print('Error de inferencia controlado:', e)

Error de inferencia controlado: La solicitud contiene valores nulos.
